# TP1 : Analyse Exploratoire de Données (EDA) sous Python

### Identification de l'étudiant

In [ ]:
# Nom : 
# Prénom : 
# Numéro d'étudiant : 

### Modalités de rendu

1. Chaque étudiant doit rendre un travail individuel.
2. Renommez ce fichier selon la convention : `TP1_Nom_Prenom.ipynb`.
3. Le rendu s'effectuera via le lien de dépôt communiqué par votre enseignant.
4. Assurez-vous que votre code s'exécute sans erreur (Menu : Kernel > Restart & Run All).

### Objectifs de la séance

L'application d'algorithmes d'apprentissage statistique nécessite le chargement, le nettoyage et l'analyse préalable des bases de données. Ce TP introduit les outils standards de l'écosystème Python :
- `pandas` : manipulation des structures de données.
- `matplotlib` et `seaborn` : visualisation statistique.

La séance comporte deux parties parallèles :
1. **Partie 1 (Guidée)** : Étude du jeu de données *Palmer Penguins*.
2. **Partie 2 (Application)** : Étude du jeu de données *Titanic*.

### Documentation utile
- [Pandas User Guide](https://pandas.pydata.org/docs/user_guide/index.html)
- [Seaborn Examples](https://seaborn.pydata.org/examples/index.html)
- [Scikit-Learn OpenML](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html)

---

## Partie 1 : Étude du jeu de données Palmer Penguins

Le jeu de données Palmer Penguins contient des relevés biométriques concernant trois espèces de manchots observées en Antarctique.

![Espèces de manchots](https://allisonhorst.github.io/palmerpenguins/reference/figures/lter_penguins.png)

### 1.1 Importation des modules

Exécutez la cellule suivante pour charger les librairies nécessaires. Nous utiliserons `fetch_openml` pour récupérer les données.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml

# Configuration pour définir le style graphique
sns.set_theme(style="whitegrid", palette="muted")

### 1.2 Chargement des données

La fonction `fetch_openml` permet de télécharger des bases de données standards.
Nous cherchons le dataset avec l'ID **42585**.

In [ ]:
# Complétez les points de suspension
# L'argument parser='auto' est recommandé pour les versions récentes de sklearn
penguins_raw = fetch_openml(data_id=..., as_frame=True, parser='auto')

# On extrait le DataFrame pandas de l'objet résultat
df_penguins = penguins_raw.frame

### 1.3 Inspection des données

Il est nécessaire de vérifier que les données ont été correctement chargées en affichant les premières lignes et les dimensions du tableau.

In [ ]:
# Affichez les 5 premières lignes avec la méthode head()
display(df_penguins....())

# Affichez les dimensions (nombre de lignes, nombre de colonnes) via l'attribut shape
print("Dimensions :", df_penguins....)

Utilisez la méthode `info()` pour inspecter le type des variables (float, int, category) et repérer le nombre de valeurs non-nulles.

In [ ]:
# Affichez les informations techniques du DataFrame
df_penguins....()

Utilisez la méthode `describe()` pour obtenir les statistiques descriptives (moyenne, écart-type, quartiles) des variables numériques.

In [ ]:
# Affichez les statistiques descriptives
display(df_penguins....())

### 1.4 Manipulation et filtrage

Nous souhaitons isoler un sous-groupe spécifique : les manchots de l'île **'Torgersen'** dont la masse corporelle (**body_mass_g**) est strictement supérieure à **4000** grammes.

In [ ]:
# Complétez les conditions de filtrage
# Rappel : pour le ET logique, utilisez l'opérateur &
gros_manchots = df_penguins[(df_penguins['island'] == ...) & (df_penguins['body_mass_g'] > ...)]

display(gros_manchots.head())

### 1.5 Traitement des valeurs manquantes

Les modèles mathématiques ne gèrent pas les données manquantes (NaN). Commençons par les compter.

In [ ]:
# Utilisez isna() suivi de sum() pour compter les NaN par colonne
print(df_penguins....()....())

Nous allons imputer (remplacer) les valeurs manquantes de la masse par la médiane, qui est moins sensible aux valeurs extrêmes que la moyenne.

In [ ]:
# Calculez la médiane de la colonne 'body_mass_g'
mediane_masse = df_penguins['body_mass_g']....()
print(f"La médiane de la masse est : {mediane_masse}")

# Remplacez les NaN de cette colonne par la médiane avec fillna()
df_penguins['body_mass_g'] = df_penguins['body_mass_g'].fillna(...)

Pour les autres colonnes contenant encore des valeurs manquantes (comme le sexe), nous supprimons simplement les lignes concernées.

In [ ]:
# On ne conserve que 'MALE' et 'FEMALE', tout le reste (comme '_') devient NaN
df_penguins['sex'] = df_penguins['sex'].where(df_penguins['sex'].isin(['MALE', 'FEMALE']), np.nan)
# Supprimez les lignes avec des valeurs manquantes restantes
df_clean = df_penguins.dropna()

print("Dimensions avant nettoyage :", df_penguins.shape)
print("Dimensions après nettoyage :", df_clean.shape)

### 1.6 Ingénierie des caractéristiques (Feature Engineering)

Il est souvent utile de créer de nouvelles variables à partir des existantes pour aider les algorithmes d'apprentissage. Nous allons catégoriser les manchots selon leur masse (Léger, Moyen, Lourd) en utilisant la méthode `apply`.

In [ ]:
# Fonction de catégorisation
def categoriser_poids(masse):
    if masse < 3500:
        return 'Léger'
    elif masse <= 4500:
        return 'Moyen'
    else:
        return 'Lourd'

# Application de la fonction sur la colonne 'body_mass_g' via une fonction lambda ou directement
df_clean['weight_category'] = df_clean['body_mass_g'].apply(lambda x: ...)

display(df_clean[['species', 'body_mass_g', 'weight_category']].head())

Une méthode plus efficace et vectorisée (propre à pandas) pour découper une variable continue en catégories est la fonction `pd.cut`. Elle évite d'écrire des fonctions manuelles et est beaucoup plus rapide sur de grands jeux de données.

In [ ]:
# Définition des bornes (bins) : 0 -> 3500 -> 4500 -> 6500
bins = [0, 3500, 4500, 6500]
labels = ['Léger', 'Moyen', 'Lourd']

# Utilisez pd.cut avec les arguments x, bins et labels
df_clean['weight_cut'] = pd.cut(df_clean['body_mass_g'], bins=..., labels=...)

display(df_clean[['body_mass_g', 'weight_category', 'weight_cut']].head())

### 1.7 Détection des valeurs aberrantes (Outliers)

Les valeurs extrêmes peuvent fausser l'apprentissage statistique. Une méthode robuste pour les détecter est l'écart interquartile (IQR).
Nous considérons comme outlier toute valeur située en dehors de l'intervalle : [Q1 - 1.5*IQR, Q3 + 1.5*IQR].

In [ ]:
# Calcul des quartiles sur la longueur du culmen
Q1 = df_clean['culmen_length_mm'].quantile(0.25)
Q3 = df_clean['culmen_length_mm'].quantile(0.75)
IQR = Q3 - Q1

# Définition des bornes
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

# Filtrage pour afficher uniquement les outliers
outliers = df_clean[(df_clean['culmen_length_mm'] < ...) | (df_clean['culmen_length_mm'] > ...)]

print(f"Nombre d'outliers détectés : {len(outliers)}")

### 1.8 Standardisation des données

Certains algorithmes (comme les k-plus proches voisins) sont sensibles à l'échelle des données. Il est courant de standardiser les variables numériques (moyenne = 0, écart-type = 1).
Formule du Z-score : z = (x - moy) / std

**Note Mathématique**: Pandas utilise par défaut l'estimateur sans biais de la variance ($n-1$).

In [ ]:
# Calcul de la moyenne et de l'écart-type de la masse
mean_mass = df_clean['body_mass_g'].mean()
std_mass = df_clean['body_mass_g'].std()

# Création d'une nouvelle colonne standardisée
df_clean['body_mass_g_scaled'] = (df_clean['body_mass_g'] - ...) / ...

# Vérification : la nouvelle moyenne doit être proche de 0
print("Moyenne après standardisation :", df_clean['body_mass_g_scaled'].mean())

### 1.9 Visualisation statistique avancée

Exploitons la puissance de `seaborn` pour générer des graphiques complexes en une ligne.

Le **Jointplot** permet de visualiser simultanément la relation entre deux variables (nuage de points) et leur distribution individuelle (histogrammes sur les côtés).

Observons la corrélation entre deux dimensions physiques : longueur et profondeur du bec (culmen).

![Culmen](https://allisonhorst.github.io/palmerpenguins/reference/figures/culmen_depth.png)

In [ ]:
plt.figure(figsize=(8, 8))

# Utilisez sns.jointplot avec x='culmen_length_mm', y='culmen_depth_mm', hue='species'
sns.jointplot(data=df_clean, x=..., y=..., hue=..., kind='scatter')

plt.show()

Le **Violinplot** (avec l'option `split=True`) est idéal pour comparer des distributions. Ici, nous comparons la masse corporelle des Mâles vs Femelles pour chaque espèce.

In [ ]:
plt.figure(figsize=(10, 6))

# Utilisez sns.violinplot avec x='species', y='body_mass_g', hue='sex'
# Ajoutez l'argument split=True pour diviser le violon en deux
sns.violinplot(data=df_clean, x=..., y=..., hue=..., split=True, inner="quart", palette="Pastel1")

plt.title("Distribution de la masse : Comparaison Mâle / Femelle")
plt.show()

Le **Relplot** (Relational Plot) est très puissant pour créer des "facettes" (Small Multiples). Il permet de diviser le graphique en plusieurs sous-graphiques selon une variable catégorielle (ici l'île) grâce à l'argument `col`.

In [ ]:
# Utilisez sns.relplot avec x='culmen_length_mm', y='culmen_depth_mm', hue='species'
# Ajoutez l'argument col='island' pour créer une colonne de graphique par île
sns.relplot(data=df_clean, x=..., y=..., hue=..., col=...)

plt.show()

Enfin, le **Pairplot** offre une vue d'ensemble de toutes les corrélations numériques du jeu de données.

In [ ]:
# Attention : cela peut prendre quelques secondes à générer
sns.pairplot(df_clean, hue=..., corner=True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

# Utilisez sns.histplot. Paramètres clés : data, x, hue, kde=True
sns.histplot(data=df_clean, x=..., hue=..., kde=True)

plt.title("Distribution de la longueur des nageoires")
plt.show()

### 1.10 Transformation des variables catégorielles

Les algorithmes de Machine Learning nécessitent des entrées numériques. Nous devons encoder les variables textuelles.

1. **Encodage binaire** pour le sexe (`map`).
2. **One-Hot Encoding** pour l'île (`get_dummies`).

In [ ]:
# Transformation manuelle avec map : FEMALE devient 0, MALE devient 1
df_clean['sex_encoded'] = df_clean['sex'].map({'FEMALE': ..., 'MALE': ...})

# Création de variables indicatrices pour 'island' (drop_first=True évite la redondance)
df_ml_ready = pd.get_dummies(df_clean, columns=[...], drop_first=True)

display(df_ml_ready.head())

---

## Partie 2 : Étude du jeu de données Titanic

Appliquez les méthodes étudiées dans la première partie sur le jeu de données historique du Titanic. L'objectif est de préparer la variable cible `survived`.

### 2.1 Chargement et inspection

Chargez les données (identifiant OpenML : **40945**) et inspectez la structure du tableau.

In [ ]:
# Chargez le dataset Titanic
titanic_raw = fetch_openml(data_id=..., as_frame=True, parser='auto')
df_titanic = titanic_raw.frame

# Affichez les premières lignes
display(df_titanic....())

Affichez les informations sur les types de colonnes. Notez que `survived` est probablement de type objet/catégorie et non numérique.

In [ ]:
# Affichez les infos (types, NaN)
df_titanic....()

### 2.2 Nettoyage des données

La variable `survived` doit être numérique pour nos calculs (0 = décédé, 1 = survécu). Certaines colonnes comme le nom ou le numéro de ticket ne sont pas pertinentes pour une analyse statistique globale.

In [ ]:
# Conversion de 'survived' en numérique
df_titanic['survived'] = pd.to_numeric(df_titanic['survived'])

# Liste des colonnes à supprimer
colonnes_a_supprimer = ['name', 'ticket', 'cabin', 'boat', 'body', 'home.dest']

# Suppression des colonnes (utilisez drop avec l'argument columns)
df_titanic = df_titanic.drop(columns=...)

# Vérification des dimensions après suppression
print("Dimensions :", df_titanic.shape)

Traitez maintenant les valeurs manquantes de la variable `age`. Remplacez les `NaN` par la médiane des âges.

In [ ]:
# Calcul de la médiane de l'âge
mediane_age = df_titanic['age']....()

# Remplacement des NaN par la médiane
df_titanic['age'] = df_titanic['age'].fillna(...)

# Suppression des dernières lignes incomplètes (s'il reste des NaN dans d'autres colonnes)
df_titanic_clean = df_titanic....()

print("Dimensions finales :", df_titanic_clean.shape)

### 2.3 Manipulation et encodage

Transformez la variable `sex` en binaire (0 pour male, 1 pour female) et appliquez un encodage One-Hot sur la variable `embarked` (port d'embarquement).

In [ ]:
# Encodage du sexe avec map
# Important : On force le type 'int' pour que le sexe apparaisse dans la matrice de corrélation
df_titanic_clean['sex'] = df_titanic_clean['sex'].map({'male': ..., 'female': ...}).astype(int)

# Encodage de 'embarked' avec get_dummies
df_titanic_ml = pd.get_dummies(df_titanic_clean, columns=[...], drop_first=True)

display(df_titanic_ml.head())

### 2.4 Visualisation

Commencez par un diagramme en barres comptant le nombre de survivants.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_titanic_clean, x=...)
plt.title("Nombre de survivants (0=Non, 1=Oui)")
plt.show()

Tracez l'histogramme des âges. 

**Note importante :** Vous observerez un pic artificiel important au centre. C'est normal : il correspond à la valeur médiane que nous avons utilisée pour remplacer toutes les données manquantes.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=df_titanic_clean, x=..., hue=..., kde=True)
plt.title("Distribution des âges (Pic dû à l'imputation par la médiane)")
plt.show()

Utilisez un tableau croisé pour analyser le taux de survie selon le sexe et la classe.

In [ ]:
tableau_survie = pd.pivot_table(df_titanic_clean, values=..., index=..., columns=..., aggfunc='mean')
display(tableau_survie)

**Analyse :** Double-cliquez ici pour écrire votre conclusion sur l'impact du sexe et de la classe sociale.

### 2.5 Matrice de corrélation

Pour interpréter correctement la matrice, voici la signification des variables moins évidentes :
*   `sibsp` : Nombre de frères, sœurs, époux ou épouses à bord.
*   `parch` : Nombre de parents ou enfants à bord.
*   `pclass` : Classe du billet (1 = 1ère classe, 3 = 3ème classe).

Assurez-vous que la variable `sex` a bien été transformée en nombres (0/1) pour apparaître dans le graphique.

In [ ]:
# Sélection des variables numériques (incluant le sexe encodé)
vars_numeriques = df_titanic_ml.select_dtypes(include=[np.number]).columns

plt.figure(figsize=(10, 8))
sns.heatmap(df_titanic_ml[vars_numeriques].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matrice de corrélation - Titanic")
plt.show()

**Analyse :** Double-cliquez ici pour identifier les variables les plus corrélées à 'survived'.